# Notebook 3 — Packaging Logic

**Series:** Python Prerequisites for Neuromatch Computational Neuroscience
**Audience:** Complete beginner — brilliant, no prior Python, no neuroscience assumed
**What this notebook covers:** Functions · Lambda functions · Importing modules · f-strings

---

> **How to use this notebook.**
> Read each explanation, then run every code cell yourself. For exercises, write your
> prediction *before* you run — the gap between prediction and result is where learning happens.
> Sections marked **Going deeper** are optional on a first pass; skip them and come back later.

---

## What you will be able to do by the end

- Write reusable, well-documented functions that accept inputs and return outputs.
- Write a quick one-line lambda function for short transformations.
- Import ready-made tools from Python's standard library and from third-party packages.
- Embed variables and expressions directly into formatted text using f-strings.

These four skills are the packaging layer of Python: they let you wrap up logic,
reuse it, borrow power from others, and present results cleanly.


---
# Item 1 — Functions

> **By the end of this item you'll be able to:** write a function with the `def` keyword,
> give it parameters with sensible defaults, call it with positional or keyword arguments,
> and get a value back with `return`. Everything below the core sections is optional on a
> first pass.


## What a function really is

**A recipe card.** A recipe for banana bread doesn't make bread; it is a set of
instructions. When you decide to bake, you follow the recipe and produce a loaf. You can
follow the same recipe twenty times with different bananas and still get a loaf each time.
The *instructions* and the *doing* are separate things.

**A hospital referral form.** Your outpatient department has a standard referral form with
fields: patient name, referring consultant, urgency level (default: "routine"), and notes.
Most of the time the form goes out with "routine" already filled in, but you can override
it to "urgent" when needed. The blank form is the definition; filling it in and sending it
is the call.

A **function** in Python is exactly a recipe or a blank form: you write the instructions
once (`def`), give it named slots for inputs (**parameters**), and call it as many times
as you like with different values (**arguments**). Each call follows the instructions and
hands back a result (`return`).

This is the most powerful idea in programming: **write once, run many times with different
inputs, get predictable outputs.** Every piece of science depends on this; no one rewrites
the formula for firing rate from scratch every time they need it.


## The anatomy of a function

```
def function_name(parameter1, parameter2):
    # indented body — the instructions
    result = parameter1 + parameter2
    return result
```

- `def` tells Python: "I am defining a new function."
- The name follows the same rules as variable names (snake_case by convention).
- **Parameters** are the named slots inside the parentheses; they are local placeholders
  that only exist while the function runs.
- The colon at the end of the `def` line is required; everything indented beneath it is
  the body.
- `return` sends a value back to whoever called the function and ends execution immediately.
  A function without a `return` statement hands back `None` (Python's "nothing here" value).


## Positional vs keyword arguments

When you *call* a function, you supply the actual values for its parameters. You can do
this in two ways.

**Positional:** values are matched to parameters left to right, by position.
```python
compute_firing_rate(20, 2.0)   # 20 -> spike_count, 2.0 -> duration
```

**Keyword:** you name the parameter explicitly, so order doesn't matter.
```python
compute_firing_rate(duration=2.0, spike_count=20)   # same result
```

The hospital form analogy again: positional is handing someone a list of answers in the
right order; keyword is filling in each box by its label. Keyword arguments make code
self-documenting and prevent order-mix errors, which matters when functions have many
parameters.

## Default parameter values

A parameter can have a **default** so the caller doesn't have to supply it every time.

```python
def threshold_check(voltage, threshold=-55):
    ...
```

Here `threshold` defaults to `-55` (millivolts — the typical voltage at which a neuron
fires). A caller who just writes `threshold_check(-60)` gets the standard threshold
automatically. A caller who needs a different threshold writes `threshold_check(-60, -50)`.

Defaults must come *after* any parameters without defaults in the definition line.


## Worked example

*(Neuroscience aside for first-time readers: a **neuron** is a brain cell that communicates
by firing brief electrical pulses called **spikes** or **action potentials**. The
**firing rate** is simply how many spikes happen per second, measured in **Hz** (hertz =
events per second). The **membrane potential** is the electrical voltage across the cell
wall, normally about -70 mV at rest; it rises toward a **threshold** voltage, around
-55 mV, at which the cell fires.)*


In [ ]:
# ------------------------------------------------------------------
# FUNCTION 1: compute_firing_rate
# A neuron fired spike_count times in duration seconds.
# Firing rate = spikes / time, the fundamental measure of neural activity.
# ------------------------------------------------------------------

def compute_firing_rate(spike_count, duration):
    """Return the firing rate in Hz (spikes per second).

    Parameters
    ----------
    spike_count : int or float
        Number of spikes (electrical pulses) recorded.
    duration : float
        Recording window in seconds.

    Returns
    -------
    float
        Firing rate in Hz.
    """
    # Guard against division-by-zero; duration must be positive
    if duration <= 0:
        raise ValueError("duration must be positive (got {})".format(duration))

    rate = spike_count / duration   # Hz = spikes / seconds
    return rate                     # send the result back to the caller

# Calling the function with positional arguments (order matters here)
rate1 = compute_firing_rate(20, 2.0)
print("Firing rate:", rate1, "Hz")   # 10.0 Hz

# Calling with keyword arguments (order doesn't matter)
rate2 = compute_firing_rate(duration=0.5, spike_count=15)
print("Firing rate:", rate2, "Hz")   # 30.0 Hz


In [ ]:
# ------------------------------------------------------------------
# FUNCTION 2: classify_neuron
# Takes a firing rate and returns a plain-English label.
# ------------------------------------------------------------------

def classify_neuron(rate):
    """Classify a neuron as 'silent', 'low', 'moderate', or 'high' firing."""
    if rate == 0:
        return "silent"
    elif rate < 10:
        return "low firing"
    elif rate < 40:
        return "moderate firing"
    else:
        return "high firing"

# Try it across a range of rates
for spikes, window in [(0, 1.0), (5, 1.0), (25, 1.0), (80, 1.0)]:
    r = compute_firing_rate(spikes, window)   # reuse the function we already defined
    label = classify_neuron(r)
    print("Spikes:", spikes, "| Rate:", r, "Hz | Class:", label)


In [ ]:
# ------------------------------------------------------------------
# FUNCTION 3: threshold_check
# voltage  : the current membrane voltage in millivolts (mV)
# threshold: the voltage at which the neuron fires (default -55 mV)
# Returns a boolean: True if the neuron will fire, False otherwise.
# ------------------------------------------------------------------

def threshold_check(voltage, threshold=-55):
    """Return True if voltage >= threshold (neuron fires)."""
    return voltage >= threshold   # comparison expression becomes the return value

# Using the default threshold
print(threshold_check(-60))    # -60 < -55, so False (neuron stays quiet)
print(threshold_check(-55))    # -55 == -55, fires! -> True
print(threshold_check(-50))    # -50 > -55, fires -> True

# Overriding the default for a different neuron type
print(threshold_check(-50, threshold=-45))  # False: -50 is below -45


---

> ## Going deeper (optional on a first pass)
>
> You can skip this block entirely the first time through and lose nothing essential.
>
> **Docstrings.** The triple-quoted string immediately inside a function body is its
> **docstring** — a built-in help message. `help(compute_firing_rate)` prints it.
> Every function you write for real work should have one. The standard formats are
> NumPy style (shown above) and Google style.
>
> **Scope (local vs global).** Variables created inside a function exist only while
> it runs; they are **local**. They vanish when the function returns. Variables
> defined outside a function are **global** and readable inside, but you cannot
> accidentally overwrite a global from inside a function without the `global` keyword
> (which you should almost never use). This isolation is a feature: it stops functions
> from stepping on each other's data.
>
> **None return.** A function that finishes without hitting a `return` statement quietly
> hands back `None`. This catches people out when they forget the `return` and then
> try to use the result. If `result = my_function(x)` and `result` is `None`, the
> first thing to check is whether the function actually `return`s.
>
> **`*args` — accepting any number of positional arguments.** The `*` syntax collects
> all extra positional arguments into a tuple.
>
> ```python
> def total_spikes(*counts):
>     return sum(counts)   # counts is a tuple of whatever was passed in
>
> print(total_spikes(5, 8, 3, 12))   # 28
> ```
>
> **`**kwargs` — accepting any number of keyword arguments.** The `**` syntax collects
> all extra keyword arguments into a dictionary.
>
> ```python
> def neuron_report(**properties):
>     for key, value in properties.items():
>         print(key, "->", value)
>
> neuron_report(type="pyramidal", region="cortex", rate_hz=25)
> ```
>
> `*args` and `**kwargs` are used in library code and wrappers; you will see them in
> NumPy and Matplotlib. You do not need to write them often, but you must be able to
> read them.


## Common questions and confusions

**"What is the difference between a parameter and an argument?"**
A **parameter** is the named slot in the `def` line (the blank on the form). An
**argument** is the actual value you supply when you call the function (what you write in
the blank). The distinction matters for error messages: Python says "takes 2 positional
arguments but 3 were given."

**"What happens if I forget `return`?"**
The function runs, does its work, then silently hands back `None`. If you then try to do
arithmetic with the result, you'll get a `TypeError: unsupported operand type(s) for +:
'NoneType' and 'int'`. That is your signal: check whether `return` is present.

**"Can a function return more than one value?"**
Yes. `return a, b` hands back a tuple `(a, b)`. You can unpack it on the calling side:
`x, y = my_function(...)`.

**"Can I call a function before I define it?"**
No. Python reads top to bottom; a name must exist before it is used. Define functions
above the code that calls them.

**"Do default values get re-evaluated every call?"**
No, and this causes a famous bug. **Never use a mutable object (list, dict) as a default
value.** The object is created once, when the `def` line is first read, and then shared
across every call. Use `None` as the default and create the list inside the body.


## Your exercises

Predict each answer **first**, then run and check.

1. Write a function `seconds_to_ms(seconds)` that converts seconds to milliseconds
   (multiply by 1000). Call it with `0.05` and print the result.

2. Write a function `spike_density(spike_count, area_mm2)` that returns spikes per mm².
   What happens if you call it with keyword arguments in reversed order?

3. Write a function `is_depolarised(voltage, threshold=-55)` that returns `True` if the
   voltage is above threshold. Test it with voltages -70, -55, -40, using both the default
   threshold and an overridden one.

4. Write a function `neuron_summary(name, rate, region="cortex")` that prints a line like:
   `"Neuron alpha | 25 Hz | cortex"`. Call it twice: once supplying all three arguments,
   once letting `region` use its default.

5. Write a function `clamp(value, low, high)` that returns `low` if `value < low`,
   `high` if `value > high`, and `value` otherwise. This is useful for keeping a voltage
   inside a realistic range. Test it with several values.

6. *(Stretch.)* Write a function `firing_statistics(spike_counts)` that accepts a **list**
   of spike counts, computes the mean and maximum, and returns both (as a tuple). Call it
   with `[5, 12, 7, 0, 20, 3]` and unpack the result into two variables, then print them.


In [ ]:
# Exercise 1
# your code here


In [ ]:
# Exercise 2
# your code here


In [ ]:
# Exercise 3
# your code here


In [ ]:
# Exercise 4
# your code here


In [ ]:
# Exercise 5
# your code here


In [ ]:
# Exercise 6 (Stretch)
# your code here


## The irreducible core

1. `def name(params):` defines a function; `return value` sends the result back.
2. **Parameters** are the slots in the definition; **arguments** are the values you
   supply at the call.
3. **Keyword arguments** (`f(x=5)`) make calls self-documenting and order-independent.
4. **Default values** (`def f(x, y=10)`) let callers omit routine inputs; defaults must
   come after required parameters.
5. A missing `return` silently gives `None` — this is the most common beginner function bug.

**You've got it when:** you can write `compute_firing_rate(spike_count, duration=1.0)`,
call it both ways, and know immediately why a result came back as `None`.


---
# Item 2 — Lambda Functions

> **By the end of this item you'll be able to:** write a one-line lambda function,
> explain when it is cleaner than a full `def`, and use it as an argument to `sorted()`.
> Everything below the core is optional on a first pass.


## What a lambda is

**A sticky-note instruction.** When a colleague asks "how do I convert this voltage reading
to millivolts?" you might scribble on a sticky note: *multiply by 1000*. You do not write
a formal protocol document for something this small. The sticky note is not stored in the
manual; it lives on the screen for a minute and gets thrown away. That is a lambda: a
quick, disposable, one-line instruction that you hand directly to something else, without
giving it a permanent name.

**A verbal instruction in the ward.** "When you sort these patient files, go by surname."
You do not write a policy document; you give the sorting rule verbally, in the moment, for
this one task. Lambda functions are verbal instructions for Python.

Formally: a **lambda** is an **anonymous function** — a function with no name, written in
a single expression. The syntax is:

```python
lambda parameter1, parameter2: expression_that_is_returned
```

There is no `def`, no name, no `return` keyword, and no body — just the expression itself.
The expression is evaluated and returned automatically.

Compare:
```python
# Full def — named, multi-line capable, reusable everywhere
def double(x):
    return x * 2

# Lambda — anonymous, one expression, usually used inline
double_lambda = lambda x: x * 2
```

Both do the same thing. The lambda version only makes sense when the function is short,
obvious, and used in one place. If you find yourself naming a lambda or reusing it, switch
to `def`.


## When to use lambda vs def

Use **lambda** when:
- The logic is a single expression (no `if`/`else` chains, no loops).
- You only need it once, right here, as an argument to another function.
- The function is so obvious its intent is clear without a name.

Use **def** when:
- The logic takes more than one line.
- You need to reuse it more than once.
- It deserves a docstring (it's part of a real interface).
- It has a name that makes the code more readable.

The classic use-case for lambda is as the `key=` argument to `sorted()` or `min()`/`max()`:
"sort these items, but judge them by *this* property." Writing a full `def` for a
one-liner key function is like filing a formal request to borrow a pen.


## Worked example


In [ ]:
# ------------------------------------------------------------------
# LAMBDA 1: normalise a membrane voltage reading
# Normalisation = rescaling so values fall between 0 and 1.
# Formula: (value - min) / (max - min)
# Here we treat -70 mV as the floor and 40 mV as the ceiling
# (the full biological range of a neuron's voltage).
# ------------------------------------------------------------------

v_min = -70.0   # resting (quiet) voltage in millivolts
v_max = 40.0    # peak voltage during a spike (action potential)

# Lambda captures v_min and v_max from the surrounding scope
normalise = lambda v: (v - v_min) / (v_max - v_min)

print(normalise(-70))   # 0.0  -> at the floor
print(normalise(40))    # 1.0  -> at the ceiling
print(normalise(-55))   # ~0.14 -> near resting; threshold is low in the range


In [ ]:
# ------------------------------------------------------------------
# LAMBDA 2: using key= in sorted()
# We have a list of (neuron_name, firing_rate) tuples.
# We want to sort by firing rate, not by name.
# ------------------------------------------------------------------

neurons = [
    ("alpha",  25.0),
    ("beta",    5.0),
    ("gamma",  80.0),
    ("delta",  12.0),
]

# sorted() needs to know HOW to compare entries.
# key= accepts any function; for each item it returns the value to compare by.
# lambda neuron: neuron[1] means "take index 1 (the rate) as the sort key."
sorted_neurons = sorted(neurons, key=lambda neuron: neuron[1])

print("Neurons ranked by firing rate (slowest to fastest):")
for name, rate in sorted_neurons:
    print(f"  {name}: {rate} Hz")

# Reverse order (fastest first) with reverse=True
print()
sorted_desc = sorted(neurons, key=lambda neuron: neuron[1], reverse=True)
print("Fastest first:")
for name, rate in sorted_desc:
    print(f"  {name}: {rate} Hz")


---

> ## Going deeper (optional on a first pass)
>
> **`map()` — apply a function to every item in a sequence.**
> `map(function, iterable)` returns an iterator; wrap it in `list()` to see the results.
>
> ```python
> voltages = [-70.0, -55.0, -40.0, 0.0, 40.0]
> normalised = list(map(lambda v: (v - (-70)) / (40 - (-70)), voltages))
> print(normalised)   # [0.0, 0.136, 0.272, 0.636, 1.0]
> ```
>
> **`filter()` — keep only items where a function returns True.**
>
> ```python
> rates = [0, 3, 15, 0, 42, 7, 60]
> active = list(filter(lambda r: r > 0, rates))
> print(active)   # [3, 15, 42, 7, 60]
> ```
>
> In modern Python, **list comprehensions** (covered in Notebook 2) are usually preferred
> over `map` and `filter` for readability, but `map` and `filter` appear often in older
> code and library internals, so you must be able to read them.
>
> **Lambdas cannot contain statements** (assignments, loops, `return` explicitly). They
> are restricted to a single expression. This is intentional: Python's designer felt that
> encouraging complex anonymous functions leads to unreadable code.


## Common questions and confusions

**"Is a lambda faster than a def?"**
No. At runtime they produce virtually identical bytecode. Choose between them for
readability, not performance.

**"Can a lambda have a default argument?"**
Yes: `lambda x, y=10: x + y`. Rarely needed; if you need defaults, a `def` is clearer.

**"Why can't I put an if/else chain in a lambda?"**
Statements (like `if:` blocks) are not allowed; only expressions. A ternary (one-liner
conditional) is allowed: `lambda x: "fire" if x >= -55 else "quiet"`. But if the logic
grows beyond a ternary, write a `def`.

**"If I name a lambda, what's the point?"**
Almost none. `double = lambda x: x * 2` is strictly worse than `def double(x): return
x * 2` — the `def` version has a proper name in tracebacks, supports docstrings, and is
what every Python style guide recommends. Name lambdas only in short tutorial illustrations
(like this one).


## Your exercises

1. Write a lambda that takes a spike count and a duration and returns the firing rate.
   Call it with `(15, 0.5)`.

2. You have this list of neuron records:
   `[("PV", 80.0), ("SST", 12.0), ("VIP", 25.0), ("PYR", 6.0)]`
   Sort it alphabetically by name using `sorted()` with a `key=` lambda.

3. Write a lambda `above_threshold` that returns `True` if a voltage is >= -55.
   Test it on -70, -55, and -40.

4. Use `map()` with a lambda to convert this list of voltages from millivolts to volts
   (divide by 1000): `[-70.0, -55.0, 0.0, 40.0]`. Print the result as a list.

5. *(Stretch.)* You have `rates = [0, 5, 0, 22, 0, 8, 60]`. Use `filter()` with a lambda
   to remove the zeros, then use `map()` with a lambda to normalise the remaining rates
   so the maximum is 1.0 (divide each by the max). Print the result.


In [ ]:
# Exercise 1
# your code here


In [ ]:
# Exercise 2
# your code here


In [ ]:
# Exercise 3
# your code here


In [ ]:
# Exercise 4
# your code here


In [ ]:
# Exercise 5 (Stretch)
# your code here


## The irreducible core

1. `lambda params: expression` is an anonymous function — no name, no `return`, one line.
2. The expression *is* the return value; no `return` keyword needed.
3. Use lambda for short, single-use, inline functions; use `def` for anything else.
4. The canonical use-case is `key=` in `sorted()`, `min()`, `max()`.
5. Lambdas cannot contain statements — only a single expression.

**You've got it when:** you can replace `def get_rate(n): return n[1]` used once as a sort
key with an equivalent lambda, and explain in one sentence why the lambda is appropriate
there but `def` would be better if you needed it in three places.


---
# Item 3 — Importing Modules

> **By the end of this item you'll be able to:** import a module with `import`,
> give it an alias with `as`, pull one specific tool out with `from ... import`,
> and explain the difference between Python's built-in standard library and
> third-party packages you install separately. The Going deeper block covers how Python
> finds modules and the `__name__` guard.


## What a module is

**Borrowing a specialist textbook.** Your hospital library holds thousands of reference
books. When you need to calculate a drug dosage, you go to the pharmacology shelf,
pull out the relevant book, and use the formula inside. You do not memorise all of
pharmacology; you borrow on demand. A **module** is a book on Python's shelf: a file
full of ready-made functions and constants. `import math` is pulling the mathematics
book off the shelf and putting it on your desk.

**Calling in a consultant.** You do not need a cardiologist every day, but when you do,
you ring one. They arrive with specialist knowledge you don't carry yourself. You refer
to them by name: "Dr. Math, what is the square root of 144?" In Python:
`math.sqrt(144)`. The dot is the "Dr." — it tells Python who you are addressing and
which function you want from them.

A **module** is simply a Python file containing definitions (functions, constants,
classes) that someone else wrote and you can reuse.


## Three ways to import

**1. `import module_name`**
Brings the whole module in under its name. You then address everything with the dot:
`math.pi`, `math.sqrt(4)`. This is the safest form: you always know where a name came
from.

**2. `import module_name as alias`**
Same as above, but you give it a shorter name. `import numpy as np` is the universal
convention in scientific Python — every textbook, paper, and tutorial on the planet uses
`np`. Typing `np.array(...)` instead of `numpy.array(...)` saves keystrokes across
thousands of lines.

**3. `from module_name import specific_name`**
Pulls one thing directly into your namespace so you can use it without the dot prefix:
`from math import sqrt` lets you write `sqrt(9)` instead of `math.sqrt(9)`. Useful when
you use one thing constantly. Avoid `from module import *` (imports everything, clutters
your namespace, and makes it impossible to tell where a name came from).


## The standard library vs third-party packages

**Standard library:** modules that ship with Python itself — no installation needed.
Examples: `math` (mathematical functions), `random` (random-number generation),
`os` (operating system tools), `datetime` (date and time). Available the instant Python
is installed.

**Third-party packages:** code other people have written and published. You install them
separately, usually with `pip install package_name` at the command line. NumPy, Matplotlib,
SciPy — everything in this course series beyond the standard library — are third-party.
Once installed, you import them exactly like standard-library modules.

Think of the standard library as the textbooks that come with medical school, and
third-party packages as specialist journals and tools you order separately as your practice
develops.


## Worked example


In [ ]:
# ------------------------------------------------------------------
# 1. import math -- standard library, always available
# ------------------------------------------------------------------
import math

# math.pi is a constant (not a function -- no parentheses needed)
print("pi:", math.pi)

# math.exp(x) computes e^x  (e is Euler's number, ~2.718)
# In neuroscience this describes exponential decay:
# how quickly a signal fades after a stimulus.
# e.g. a synaptic current decays with time constant tau (tau = how fast it fades)
tau = 0.020    # 20 milliseconds, a typical synaptic time constant
t   = 0.010    # 10 ms after the peak

decay = math.exp(-t / tau)   # fraction of peak current remaining
print(f"Fraction of synaptic current remaining at {t*1000:.0f} ms: {decay:.4f}")
# Should be ~0.607 (about 60% remains after one time-constant)

print("Square root of 144:", math.sqrt(144))
print("log(1):", math.log(1))   # natural log; math.log10() for base-10


In [ ]:
# ------------------------------------------------------------------
# 2. from math import sqrt -- pull one function in directly
# ------------------------------------------------------------------
from math import sqrt, pi

# Now we can call sqrt() and pi without the math. prefix
radius = 5.0
circumference = 2 * pi * radius
print(f"Circumference of a circle with radius {radius}: {circumference:.4f}")

hypotenuse = sqrt(3**2 + 4**2)
print("Hypotenuse of 3-4-5 triangle:", hypotenuse)


In [ ]:
# ------------------------------------------------------------------
# 3. random module -- simulating neural data
# ------------------------------------------------------------------
import random

# random.seed() fixes the random sequence so results are reproducible.
# Think of it as telling the random-number machine where to start.
random.seed(42)

# Simulate spike counts: a neuron observed for 10 trials,
# with a typical count between 0 and 30 spikes per trial.
spike_counts = [random.randint(0, 30) for _ in range(10)]
print("Simulated spike counts:", spike_counts)

# Simulate noisy voltage measurements (Gaussian noise around resting potential)
# random.gauss(mean, std_dev) draws from a normal distribution
resting_voltages = [random.gauss(-70, 2) for _ in range(5)]
print("Noisy voltage readings (mV):")
for v in resting_voltages:
    print(f"  {v:.2f} mV")


In [ ]:
# ------------------------------------------------------------------
# 4. import numpy as np -- the universal scientific alias (preview)
# ------------------------------------------------------------------
import numpy as np   # np is the agreed alias used by everyone

# Arrays are NumPy's superpower (covered fully in Notebook 4)
spike_array = np.array([5, 12, 7, 0, 20, 3, 15])

print("Mean firing rate:", np.mean(spike_array), "spikes/trial")
print("Max spikes in a trial:", np.max(spike_array))
print("Type of spike_array:", type(spike_array))


---

> ## Going deeper (optional on a first pass)
>
> **How Python finds modules.** When you write `import math`, Python searches a list of
> directories in order: (1) the directory containing the script you are running, (2) the
> standard library directories, (3) directories listed in the `PYTHONPATH` environment
> variable, (4) installed third-party package directories. You can inspect this list with
> `import sys; print(sys.path)`.
>
> **`__name__ == "__main__"`.** Every Python file has a built-in variable `__name__`.
> When you run a file directly (`python myscript.py`), `__name__` is set to `"__main__"`.
> When a file is *imported* by another file, `__name__` is the file's own name instead.
> This means you can write:
>
> ```python
> if __name__ == "__main__":
>     # code here only runs when this file is executed directly,
>     # not when it is imported as a module
>     main()
> ```
>
> This is how library files protect their test code from running when imported.
>
> **`pip install`** is the standard command for installing third-party packages from the
> Python Package Index (PyPI). In a terminal: `pip install numpy`. In a Jupyter notebook
> or Colab: `!pip install numpy` (the `!` runs a shell command). Google Colab has NumPy,
> Matplotlib, and SciPy pre-installed, so you rarely need to install anything there.
>
> **Writing your own module** is simply saving functions in a `.py` file and then
> importing it like any other module. If `my_tools.py` sits in the same directory as
> your notebook, `import my_tools` works immediately.


## Common questions and confusions

**"What is the difference between `import math` and `from math import sqrt`?"**
The first puts the module on your desk; you address its contents as `math.sqrt`. The
second takes one book out of the module and puts it directly in front of you; you just
call `sqrt`. The first form is safer in larger projects because it is always clear where
a name came from.

**"Why does everyone alias NumPy as `np`?"**
It is an established community convention, not a rule. Every scientific paper, tutorial,
and Stack Overflow answer uses `np`. Writing anything else means your code doesn't match
the world's documentation, which is more confusing than any saved keystrokes.

**"Can I give a module any alias I like?"**
Python allows it. Convention forbids it for well-known packages: always `np` for NumPy,
`pd` for Pandas, `plt` for Matplotlib's pyplot. Use arbitrary aliases only for your own
private helpers.

**"Does importing a module run its code?"**
Yes. The module file is executed from top to bottom when first imported. Constants are
set, functions defined. Subsequent `import` statements for the same module return the
cached version without re-running it. This is why module-level side-effects (printing,
reading files) are bad practice.


## Your exercises

1. Import `math` and print `math.e` (Euler's number). Then compute and print
   `math.log(math.e)` (what should the answer be, and why?).

2. Using `math.exp`, write a one-line expression for the voltage decay formula
   `V(t) = V0 * exp(-t / tau)` where `V0 = 1.0`, `t = 0.05` seconds, `tau = 0.02`
   seconds. Print the result to 4 decimal places.

3. Use `from math import floor, ceil` to import two rounding functions. Apply them to
   `7.3` and print the results. What is the difference?

4. Import `random`, set `random.seed(99)`, then generate a list of 8 random firing rates
   using `random.uniform(0, 60)` (a float between 0 and 60 Hz). Print the list.

5. Using the list from Exercise 4, use `sorted()` with a lambda key to sort the rates in
   descending order (combining what you learned in Items 2 and 3).

6. *(Stretch.)* Import `math`. A population of neurons is described by the formula:
   `N(t) = N0 * exp(r * t)`, where `N0` is the starting count, `r` is the growth rate,
   and `t` is time in seconds. Write a function `population_at(N0, r, t)` that computes
   this. Print the population at t = 0, 1, 2, 5 seconds with N0=100 neurons and r=0.2.


In [ ]:
# Exercise 1
# your code here


In [ ]:
# Exercise 2
# your code here


In [ ]:
# Exercise 3
# your code here


In [ ]:
# Exercise 4
# your code here


In [ ]:
# Exercise 5
# your code here


In [ ]:
# Exercise 6 (Stretch)
# your code here


## The irreducible core

1. `import module` puts the module on your desk; access contents with `module.name`.
2. `import module as alias` gives it a shorter desk label; `np` for NumPy is universal.
3. `from module import name` pulls one item to your desk directly; avoid `import *`.
4. The **standard library** ships with Python; **third-party packages** are installed
   separately with `pip install`.
5. Importing a module runs it once; subsequent imports reuse the cached version.

**You've got it when:** you can look at `from math import sqrt` and `import math` and
explain exactly what each puts where, and know which form to choose for a given situation.


---
# Item 4 — f-strings

> **By the end of this item you'll be able to:** embed variables and expressions
> directly into a string using the `f"..."` syntax, control decimal places and alignment
> with format specifiers, and produce clean, readable output for neuroscience results.
> Going deeper covers multiline f-strings, nested f-strings, and a brief history of
> older formatting styles.


## What an f-string is

**A mail-merge letter template.** Every GP surgery sends the same appointment letter to
hundreds of patients, but each copy has the patient's name, date, and time filled in
automatically. The template says "Dear [NAME], your appointment is on [DATE] at [TIME]."
The clinic's system swaps in the real values. An f-string is that template: a fixed
piece of text with slots where Python swaps in your variables.

**A lab report with blanks.** A histology report form has lines like:
`"Neuron type: _____, Firing rate: _____ Hz, Region: _____"`. You fill in the blanks
each time with different measurements. An f-string is a Python line that does the same
thing automatically.

**The syntax:** put the letter `f` immediately before the opening quote, then wrap any
variable or expression in curly braces `{}` inside the string.

```python
name = "alpha"
rate = 25.4
print(f"Neuron {name} fires at {rate} Hz")
# Output: Neuron alpha fires at 25.4 Hz
```

Python evaluates whatever is in the braces — it can be a variable, a calculation, a
function call, anything — and inserts the result as text. No manual conversion with
`str()` needed.


## Format specifiers: controlling how values look

After the variable name (or expression), add a colon `:` followed by a format code.

| Specifier | Meaning | Example | Output |
|-----------|---------|---------|--------|
| `:.2f` | float, 2 decimal places | `f"{3.14159:.2f}"` | `3.14` |
| `:.4f` | float, 4 decimal places | `f"{3.14159:.4f}"` | `3.1416` |
| `:d` | integer (whole number) | `f"{42:d}"` | `42` |
| `:>10` | right-align in 10 chars | `f"{'Hz':>10}"` | `        Hz` |
| `:<10` | left-align in 10 chars | `f"{'Hz':<10}"` | `Hz        ` |
| `:^10` | centre in 10 chars | `f"{'Hz':^10}"` | `    Hz    ` |
| `:06d` | pad integer with zeros | `f"{7:06d}"` | `000007` |
| `:.2e` | scientific notation | `f"{0.00314:.2e}"` | `3.14e-03` |
| `:,` | thousands separator | `f"{1000000:,}"` | `1,000,000` |

The format specifier sits inside the braces, after a colon: `{value:specifier}`.


## Worked example


In [ ]:
# ------------------------------------------------------------------
# Basic f-string usage
# ------------------------------------------------------------------
neuron_name   = "Layer 5 pyramidal"
spike_count   = 37
duration_s    = 1.5    # duration in seconds
firing_rate   = spike_count / duration_s

# Simple embedding
print(f"Neuron type: {neuron_name}")
print(f"Spike count: {spike_count}")
print(f"Duration:    {duration_s} s")

# Expression directly inside the braces (no intermediate variable needed)
print(f"Firing rate: {spike_count / duration_s:.2f} Hz")

# Using a variable already computed
print(f"Firing rate: {firing_rate:.2f} Hz")


In [ ]:
# ------------------------------------------------------------------
# Formatting a neat neuron stats table using alignment specifiers
# ------------------------------------------------------------------
neurons = [
    ("alpha",   37, 1.5),
    ("beta",     5, 2.0),
    ("gamma",  120, 3.0),
    ("delta",    0, 1.0),
]

# Print a header line
print(f"{'Neuron':<10} {'Spikes':>8} {'Duration (s)':>14} {'Rate (Hz)':>12}")
print("-" * 48)

for name, spikes, dur in neurons:
    rate = spikes / dur
    # :<10  left-aligns the name in 10 characters
    # :>8   right-aligns integer in 8 characters
    # :>14.1f  right-aligns float, 1 decimal place, in 14 characters
    # :>12.2f  right-aligns float, 2 decimal places, in 12 characters
    print(f"{name:<10} {spikes:>8d} {dur:>14.1f} {rate:>12.2f}")


In [ ]:
# ------------------------------------------------------------------
# Simulation results: formatting scientific notation and percentages
# ------------------------------------------------------------------
import math

# Synaptic conductance values are very small (nano-siemens range)
conductance = 0.000000025   # 25 nanosiemens in SI units (siemens)

# :e gives scientific notation; :.2e means 2 decimal places
print(f"Synaptic conductance: {conductance:.2e} S")

# Percentage of trials above threshold
above = 47
total = 100
print(f"Threshold crossings: {above}/{total} ({above/total:.1%})")
# :.1%  multiplies by 100 and appends %, 1 decimal place

# Decay over time (same exponential formula as Item 3)
tau = 0.020   # 20 ms time constant
print()
print(f"{'Time (ms)':>12}  {'Remaining (%)'}")
for t_ms in [0, 10, 20, 40, 80]:
    t = t_ms / 1000.0
    fraction = math.exp(-t / tau)
    print(f"{t_ms:>12d}  {fraction * 100:>8.2f}%")


---

> ## Going deeper (optional on a first pass)
>
> **Multiline f-strings.** You can span an f-string across lines by using triple quotes
> (three single or double quotes at the start and end). Everything in between is included,
> including newlines and indentation. This is useful for building multi-line report blocks.
> Example (pseudocode — do not run this inline comment as code):
> the string starts with f and three quotes, spans multiple lines, and ends with three quotes.
>
> **Expressions, not just variables.** Any Python expression can go in the braces:
> `f"{2 ** 10}"` gives `"1024"`. You can call functions: `f"{len(spikes)}"`,
> `f"{math.sqrt(voltage):.3f}"`. You can even format conditionally:
> `f"{'FIRE' if voltage >= -55 else 'quiet'}"`.
>
> **Nested f-strings.** The format specifier itself can be an f-string or variable:
> `f"{value:.{decimals}f}"` where `decimals` is a variable. Useful when precision is
> itself a parameter.
>
> **Datetime formatting.** Python's `datetime` objects plug into f-strings with the
> same `strftime` codes: `f"{datetime.now():%Y-%m-%d %H:%M}"` prints the current date
> and time. Useful for labelling output files.
>
> **Older formatting styles (brief history).**
> Before f-strings (introduced in Python 3.6), there were two other styles:
>
> *Percent formatting (Python 2 era):*
> `"Rate: %.2f Hz" % rate` — uses `%` as the placeholder and operator.
>
> *.format() method (Python 3.0-3.5):*
> `"Rate: {:.2f} Hz".format(rate)` — same specifier syntax but called as a method.
>
> Both still work in Python 3 and appear in older code. f-strings are the modern
> standard: they are faster, more readable, and support arbitrary expressions.


## Common questions and confusions

**"Do I need the `f` before every string?"**
Only when you want Python to evaluate something in curly braces. A plain `"hello {name}"`
without the `f` is just text; the `{name}` part is printed literally without substitution.

**"What if my value already has too many decimal places and I want fewer?"**
That is exactly what `:.2f` is for: it rounds *for display* (the underlying value is
unchanged). `f"{3.14159:.2f}"` prints `"3.14"` but the number `3.14159` still exists
in memory.

**"Can I use single quotes inside an f-string?"**
Yes, as long as the outer quotes are double (or vice versa). Inside the braces you can
also use either, but keep them different from the outer quotes or escape them.

**"What does the `>` or `<` in a specifier do?"**
It controls alignment inside a fixed-width field. `f"{'yes':>10}"` pads with spaces on
the left so the text sits at the right of a 10-character slot. This is how you make
columns line up in plain-text tables.

**"Can I do maths inside the braces?"**
Yes: `f"{spike_count * 2}"` is valid. The expression is evaluated first and the result
is formatted. Keep expressions short for readability; complex logic belongs outside the
string.


## Your exercises

1. Create variables `neuron = "beta"`, `rate = 47.3829`, `region = "hippocampus"`. Print a
   sentence using an f-string: `"Neuron beta fires at 47.38 Hz in the hippocampus."` (rate
   to 2 decimal places).

2. Print a right-aligned table header and two rows for:
   - Neuron "A", 12 spikes, 0.5 s
   - Neuron "B", 80 spikes, 2.0 s
   Show the computed firing rate to 1 decimal place.

3. Format the number `0.000186` in scientific notation with 3 decimal places.

4. Using a loop over `[0, 1, 2, 5, 10]` seconds and `tau = 5.0`, print a table of time
   vs exponential decay `exp(-t/tau)` formatted to 4 decimal places.

5. Print the fraction `37 / 100` as a percentage with 1 decimal place using the `%`
   format specifier (`:,.1%`).

6. *(Stretch.)* Write a function `format_neuron_report(name, spike_count, duration_s,
   region="cortex")` that returns a formatted multi-line string (plain concatenation or
   multiple print statements are fine) showing all four fields, rate computed inside the
   function, rate to 2 decimal places, all fields aligned neatly. Call it twice with
   different inputs and print the results.


In [ ]:
# Exercise 1
# your code here


In [ ]:
# Exercise 2
# your code here


In [ ]:
# Exercise 3
# your code here


In [ ]:
# Exercise 4
# your code here


In [ ]:
# Exercise 5
# your code here


In [ ]:
# Exercise 6 (Stretch)
# your code here


## The irreducible core

1. `f"...{variable}..."` embeds a variable or expression directly into a string — no
   `str()` conversion needed.
2. Format specifiers go after a colon inside the braces: `{value:.2f}` rounds to 2
   decimal places; `{value:>10}` right-aligns in 10 characters.
3. Anything in `{}` is a Python expression and is evaluated at runtime.
4. The `f` prefix is required; without it, curly braces are printed literally.
5. f-strings are the modern standard (Python 3.6+); `.format()` and `%` are older
   alternatives you will see in existing code.

**You've got it when:** you can write `f"{firing_rate:.2f} Hz"` without looking up the
syntax and explain what each part of the specifier does.


---
# Solutions — try first!

Work through every exercise yourself before reading these. The point is the gap between
your prediction and the actual output, not the answer itself.


## Item 1 — Functions: Solutions


In [ ]:
# Solution 1: seconds_to_ms
def seconds_to_ms(seconds):
    """Convert seconds to milliseconds."""
    return seconds * 1000   # 1 second = 1000 ms

print(seconds_to_ms(0.05))   # 50.0 ms


In [ ]:
# Solution 2: spike_density with keyword arguments
def spike_density(spike_count, area_mm2):
    """Return spikes per square millimetre."""
    return spike_count / area_mm2

# Positional (normal) call
print(spike_density(50, 2.5))         # 20.0

# Keyword arguments -- order can be reversed safely
print(spike_density(area_mm2=2.5, spike_count=50))  # still 20.0


In [ ]:
# Solution 3: is_depolarised with default threshold
def is_depolarised(voltage, threshold=-55):
    """True if voltage is at or above threshold (neuron fires)."""
    return voltage >= threshold

print(is_depolarised(-70))           # False (quiet)
print(is_depolarised(-55))           # True (exactly at threshold)
print(is_depolarised(-40))           # True (well above)
print(is_depolarised(-50, threshold=-45))  # False (-50 < -45)


In [ ]:
# Solution 4: neuron_summary with default region
def neuron_summary(name, rate, region="cortex"):
    """Print a formatted one-liner for a neuron."""
    print(f"Neuron {name} | {rate} Hz | {region}")

neuron_summary("alpha", 25, "hippocampus")   # region supplied
neuron_summary("beta",  12)                  # region uses default "cortex"


In [ ]:
# Solution 5: clamp a value between low and high
def clamp(value, low, high):
    """Return value clipped to [low, high]."""
    if value < low:
        return low
    elif value > high:
        return high
    else:
        return value

print(clamp(-80, -70, 40))   # -70 (below floor)
print(clamp(-55, -70, 40))   # -55 (in range, returned as-is)
print(clamp(60,  -70, 40))   # 40  (above ceiling)


In [ ]:
# Solution 6 (Stretch): firing_statistics returning mean and max
def firing_statistics(spike_counts):
    """Return (mean, max) of a list of spike counts."""
    mean_val = sum(spike_counts) / len(spike_counts)
    max_val  = max(spike_counts)
    return mean_val, max_val   # returns a tuple

mean_spikes, max_spikes = firing_statistics([5, 12, 7, 0, 20, 3])
print(f"Mean spikes: {mean_spikes:.2f}")   # 7.83
print(f"Max spikes:  {max_spikes}")        # 20


## Item 2 — Lambda Functions: Solutions


In [ ]:
# Solution 1: lambda for firing rate
firing_rate = lambda spike_count, duration: spike_count / duration
print(firing_rate(15, 0.5))   # 30.0 Hz


In [ ]:
# Solution 2: sort neuron records alphabetically by name
neuron_types = [("PV", 80.0), ("SST", 12.0), ("VIP", 25.0), ("PYR", 6.0)]
sorted_by_name = sorted(neuron_types, key=lambda n: n[0])
print(sorted_by_name)   # [('PV', 80.0), ('PYR', 6.0), ('SST', 12.0), ('VIP', 25.0)]


In [ ]:
# Solution 3: lambda above_threshold
above_threshold = lambda v: v >= -55
print(above_threshold(-70))   # False
print(above_threshold(-55))   # True
print(above_threshold(-40))   # True


In [ ]:
# Solution 4: map + lambda to convert mV to V
voltages_mv = [-70.0, -55.0, 0.0, 40.0]
voltages_v  = list(map(lambda v: v / 1000.0, voltages_mv))
print(voltages_v)   # [-0.07, -0.055, 0.0, 0.04]


In [ ]:
# Solution 5 (Stretch): filter zeros, then normalise by max
rates = [0, 5, 0, 22, 0, 8, 60]

# Step 1: remove zeros
active = list(filter(lambda r: r > 0, rates))
print("Active rates:", active)   # [5, 22, 8, 60]

# Step 2: normalise so maximum is 1.0
max_rate   = max(active)
normalised = list(map(lambda r: r / max_rate, active))
print("Normalised:", [round(n, 4) for n in normalised])


## Item 3 — Importing Modules: Solutions


In [ ]:
# Solution 1: math.e and log
import math
print("Euler's number e:", math.e)        # ~2.71828
print("log(e):", math.log(math.e))        # 1.0 exactly -- log base e of e is always 1


In [ ]:
# Solution 2: voltage decay formula
import math
V0 = 1.0; t = 0.05; tau = 0.02
V_t = V0 * math.exp(-t / tau)
print(f"V(t={t}s) = {V_t:.4f}")   # ~0.0821 (heavily decayed, t > 2*tau)


In [ ]:
# Solution 3: floor and ceil
from math import floor, ceil
print("floor(7.3):", floor(7.3))   # 7  (round down always)
print("ceil(7.3):",  ceil(7.3))    # 8  (round up always)
# floor rounds toward negative infinity; ceil toward positive infinity


In [ ]:
# Solution 4: random firing rates
import random
random.seed(99)
rates = [random.uniform(0, 60) for _ in range(8)]
print([round(r, 2) for r in rates])


In [ ]:
# Solution 5: sort those rates descending (using lambda from Item 2)
# (run Solution 4 first to define 'rates')
desc_rates = sorted(rates, key=lambda r: r, reverse=True)
print([round(r, 2) for r in desc_rates])


In [ ]:
# Solution 6 (Stretch): population growth formula
import math

def population_at(N0, r, t):
    """Compute N(t) = N0 * exp(r * t) for exponential population growth."""
    return N0 * math.exp(r * t)

N0 = 100.0
r  = 0.2
for t in [0, 1, 2, 5]:
    N = population_at(N0, r, t)
    print(f"t = {t} s -> N = {N:.1f} neurons")


## Item 4 — f-strings: Solutions


In [ ]:
# Solution 1
neuron = "beta"
rate   = 47.3829
region = "hippocampus"
print(f"Neuron {neuron} fires at {rate:.2f} Hz in the {region}.")


In [ ]:
# Solution 2: aligned table
data = [("A", 12, 0.5), ("B", 80, 2.0)]
print(f"{'Neuron':<8} {'Spikes':>8} {'Duration':>10} {'Rate (Hz)':>12}")
print("-" * 42)
for name, spikes, dur in data:
    print(f"{name:<8} {spikes:>8d} {dur:>10.1f} {spikes/dur:>12.1f}")


In [ ]:
# Solution 3: scientific notation
value = 0.000186
print(f"{value:.3e}")   # 1.860e-04


In [ ]:
# Solution 4: exponential decay table
import math
tau = 5.0
print(f"{'t (s)':>8}  {'exp(-t/tau)':>14}")
print("-" * 26)
for t in [0, 1, 2, 5, 10]:
    print(f"{t:>8d}  {math.exp(-t / tau):>14.4f}")


In [ ]:
# Solution 5: percentage
count = 37
total = 100
print(f"{count/total:.1%}")   # 37.0%


In [ ]:
# Solution 6 (Stretch): format_neuron_report function
def format_neuron_report(name, spike_count, duration_s, region="cortex"):
    """Return a formatted multi-line neuron summary string."""
    rate = spike_count / duration_s
    lines = [
        f"Neuron:       {name}",
        f"Region:       {region}",
        f"Spike count:  {spike_count:d}",
        f"Duration:     {duration_s:.2f} s",
        f"Firing rate:  {rate:.2f} Hz",
    ]
    return "\n".join(lines)

print(format_neuron_report("alpha", 37, 1.5, region="hippocampus"))
print()
print(format_neuron_report("beta",  5, 2.0))
